<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>01. GP Classification and Kernel Engineering</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [GP Classification Overview](#-part-i-gp-classification)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Kernel Engineering](#-part-iii-kernel-engineering)**
    - 3.1 [Kernel Arithmetic: Combining Kernels](#kernel-arithmetic)
    - 3.2 [Designing Kernels for Specific Patterns](#designing-kernels)
- **4. [GP Classification with TFP](#-part-iv-gp-classification-with-tfp)**
    - 4.1 [Binary Classification](#binary-classification)
    - 4.2 [Decision Boundaries with Uncertainty](#decision-boundaries)
- **5. [Kernel Selection via Cross-Validation](#-part-v-kernel-selection)**
- **6. [Key Takeaways](#-part-vi-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. GP Classification Overview</span>

- For **regression**, GPs assume Gaussian likelihood → closed-form posterior
- For **classification**, we need a **non-Gaussian likelihood** (e.g., Bernoulli) → no closed-form posterior
- Solution: Use a **latent GP** passed through a sigmoid/softmax link function:

$$f \sim \mathcal{GP}(0, k), \quad p(y=1|x) = \sigma(f(x))$$

- Inference requires approximations: **Laplace approximation**, **EP**, or **MCMC**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions
tfk = tfp.math.psd_kernels

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Kernel Engineering</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Kernel Arithmetic: Combining Kernels</span>

Kernels can be **composed** to create richer models:

| **Operation** | **Effect** | **Example** |
|--------------|-----------|-------------|
| $k_1 + k_2$ | Sum of patterns | Trend + Seasonality |
| $k_1 \times k_2$ | Interaction of patterns | Modulated periodicity |
| $k_1 \circ g$ | Input warping | Non-stationary patterns |

In [ ]:
# Demonstrate kernel composition
x = np.linspace(-5, 5, 200).reshape(-1, 1).astype(np.float32)

# Individual kernels
rbf = tfk.ExponentiatedQuadratic(amplitude=1.0, length_scale=1.0)
periodic = tfk.ExpSinSquared(amplitude=0.5, length_scale=0.5, period=2.0)
linear = tfk.Linear()

# Compositions
kernels = {
    'RBF': rbf,
    'Periodic': periodic,
    'RBF + Periodic': rbf + periodic,  # Additive
    'RBF × Periodic': rbf * periodic,  # Multiplicative (modulated)
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, k) in zip(axes.flat, kernels.items()):
    gp = tfd.GaussianProcess(kernel=k, index_points=x, observation_noise_variance=1e-6)
    samples = gp.sample(3)
    for i in range(3):
        ax.plot(x[:, 0], samples[i].numpy(), linewidth=2, alpha=0.7)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('f(x)', fontsize=12)

plt.suptitle('Kernel Composition: Building Complex Patterns from Simple Kernels',
             fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. GP Classification with TFP</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. Binary Classification</span>

In [ ]:
# Generate 2D classification data
np.random.seed(42)
n_per_class = 50

X_pos = np.random.randn(n_per_class, 2).astype(np.float32) + np.array([2, 2])
X_neg = np.random.randn(n_per_class, 2).astype(np.float32) + np.array([-2, -2])
X_train = np.vstack([X_pos, X_neg])
y_train = np.concatenate([np.ones(n_per_class), np.zeros(n_per_class)]).astype(np.float32)

# Shuffle
perm = np.random.permutation(len(X_train))
X_train, y_train = X_train[perm], y_train[perm]

# Use GP + Bernoulli for classification via MCMC
kernel = tfk.ExponentiatedQuadratic(amplitude=1.0, length_scale=1.0)

def gpc_log_prob(f_values):
    """Log posterior for GP classification."""
    # GP prior on latent function values
    gp_prior = tfd.GaussianProcess(
        kernel=kernel,
        index_points=tf.constant(X_train),
        observation_noise_variance=1e-6
    )
    log_prior = gp_prior.log_prob(f_values)
    
    # Bernoulli likelihood through sigmoid
    probs = tf.sigmoid(f_values)
    log_lik = tf.reduce_sum(
        tfd.Bernoulli(probs=probs).log_prob(y_train)
    )
    
    return log_prior + log_lik

# MCMC sampling
@tf.function
def sample_gpc():
    kernel_mcmc = tfp.mcmc.SimpleStepSizeAdaptation(
        inner_kernel=tfp.mcmc.HamiltonianMonteCarlo(
            target_log_prob_fn=gpc_log_prob,
            step_size=0.1,
            num_leapfrog_steps=10
        ),
        num_adaptation_steps=200,
        target_accept_prob=0.75
    )
    return tfp.mcmc.sample_chain(
        num_results=500,
        num_burnin_steps=300,
        current_state=tf.zeros(len(X_train)),
        kernel=kernel_mcmc,
        trace_fn=None
    )

f_samples = sample_gpc()
print(f"Posterior function samples: {f_samples.shape}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.2. Decision Boundaries with Uncertainty</span>

In [ ]:
# Predict on a grid
xx, yy = np.meshgrid(np.linspace(-5, 5, 60), np.linspace(-5, 5, 60))
X_grid = np.stack([xx.ravel(), yy.ravel()], axis=-1).astype(np.float32)

# Compute posterior predictive probabilities
mean_f = tf.reduce_mean(f_samples, axis=0).numpy()

# Use the posterior mean function values and kernel to predict at new points
# Simple nearest-neighbor-like approach for visualization
from scipy.interpolate import griddata

# Interpolate mean latent function
f_grid = griddata(X_train, mean_f, X_grid, method='cubic', fill_value=0.0)
prob_grid = 1.0 / (1.0 + np.exp(-f_grid))

fig, ax = plt.subplots(figsize=(10, 8))
contour = ax.contourf(xx, yy, prob_grid.reshape(xx.shape), levels=20, cmap='RdBu_r', alpha=0.8)
plt.colorbar(contour, ax=ax, label='P(class=1)')
ax.contour(xx, yy, prob_grid.reshape(xx.shape), levels=[0.5], colors='black', linewidths=2)

ax.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='r', s=50, 
           edgecolors='white', linewidth=1.5, label='Class 1', zorder=5)
ax.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='b', s=50,
           edgecolors='white', linewidth=1.5, label='Class 0', zorder=5)

ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('GP Classification: Decision Boundary with Uncertainty',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| GP Classification | Latent GP + sigmoid link → probabilistic classification |
| Kernel composition | Sum, product, and warp kernels for rich patterns |
| Kernel selection | Use marginal likelihood or cross-validation |
| Inference | Requires approximation (Laplace, MCMC) unlike GP regression |
| Decision boundaries | Naturally uncertain — wider in sparse data regions |